In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import time
import numpy as np

# Thêm thư mục gốc vào path để import từ src
sys.path.append(os.path.abspath(".."))
from src.data_csp import load_pilot_data, solve_csp

print("Đã nạp module thành công!")

In [ ]:
# Tạo thư mục lưu data
os.makedirs("../data", exist_ok=True)

# 1 bảng giải chuẩn và 1 bảng đề hợp lệ 9x9
sample_sol = np.array([
    5, 3, 4, 6, 7, 8, 9, 1, 2,
    6, 7, 2, 1, 9, 5, 3, 4, 8,
    1, 9, 8, 3, 4, 2, 5, 6, 7,
    8, 5, 9, 7, 6, 1, 4, 2, 3,
    4, 2, 6, 8, 5, 3, 7, 9, 1,
    7, 1, 3, 9, 2, 4, 8, 5, 6,
    9, 6, 1, 5, 3, 7, 2, 8, 4,
    2, 8, 7, 4, 1, 9, 6, 3, 5,
    3, 4, 5, 2, 8, 6, 1, 7, 9
], dtype=np.int64)

# Tạo bảng đố bằng cách ẩn một số ô (gán bằng 0)
sample_quiz = sample_sol.copy()
mask_indices = [1, 2, 5, 10, 11, 14, 19, 20, 24, 28, 32, 40, 48, 56, 60, 66, 70, 75, 78, 80]
sample_quiz[mask_indices] = 0

num_samples = 10700
quizzes = np.tile(sample_quiz, (num_samples, 1))
solutions = np.tile(sample_sol, (num_samples, 1))

# Lưu file pilot
data_path = "../data/sudoku_pilot.npz"
np.savez_compressed(data_path, quizzes=quizzes, solutions=solutions)

print(f"Đã lưu dữ liệu tại: {data_path}")
print(f"Kích thước mảng câu đố: {quizzes.shape}")

In [ ]:
# Lấy 1 mẫu test để chạy thử giải thuật CSP
quiz_test = quizzes[0]

# Chạy giải đố bằng MRV
pred_board, step_history = solve_csp(quiz_test, method="mrv", n_size=9)

print("Số ô trống còn lại:", (pred_board == 0).sum())
print("Tổng số bước giải:", len(step_history))

if len(step_history) > 0:
    first_step = step_history[0]
    print("\nCác khóa trong step_history[0]:", list(first_step.keys()))
    # Kiểm tra Data Contract
    assert list(first_step.keys()) == ['step', 'unmasked_indices', 'values', 'margins', 'board_state']
    print("=> HỢP ĐỒNG DỮ LIỆU (DATA CONTRACT) CHUẨN XÁC!")

In [ ]:
# Lấy 10 mẫu test cuối để chạy đánh giá baseline
test_quizzes = quizzes[-10:]
preds = []

start_time = time.time()
for q in test_quizzes:
    pred, _ = solve_csp(q, method="mrv", n_size=9)
    preds.append(pred)
elapsed_time = time.time() - start_time

print(f"Thời gian giải {len(test_quizzes)} mẫu test: {elapsed_time:.4f} giây")

# Lưu file kết quả baseline cho TV4
result_path = "../baseline_csp_results.npz"
np.savez_compressed(
    result_path,
    preds=np.array(preds),
    time_taken=elapsed_time
)
print(f"Đã xuất kết quả baseline tại: {result_path}")